# Options from Zero · Episode 6: Risk-neutral probability

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The shortcut · 4. Average, then discount · 5. Why 'risk-neutral' · 6. Fake odds, real price · 7. When rates change · 8. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 6,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

Delta for a strike of \$80: the call pays \$40 if the share goes to \$120 and nothing at \$80, so delta = (40 − 0) / (120 − 80) = 1. The copy is one whole share (with a loan of the present value of \$80): the option moves one for one with the share.

In [ ]:
UP, DOWN = 120.0, 80.0
out["answer"] = {"strike": 80.0, "pay_up": UP - 80.0, "delta_80": one_step(S, UP, DOWN, 80.0, T, r)["delta"]}
out["answer"]

## 3. The shortcut

There's one set of odds under which "expected payoff, discounted" gives exactly the replication price:

$$q = \frac{e^{rT} - d}{u - d}, \qquad u = \frac{120}{100}, \; d = \frac{80}{100}$$

In [ ]:
tree = one_step(S, UP, DOWN, K, T, r)
q = tree["q"]
out["q"] = {"q": q, "q_down": 1 - q, "q_pct": q * 100, "q_down_pct": (1 - q) * 100, "growth": math.exp(r * T),
            "u": UP / S, "d": DOWN / S,
            "nodes": [{"t": 0, "k": 0, "price": S, "value": tree["price"]},
                      {"t": 1, "k": 0, "price": DOWN, "value": tree["v_down"]},
                      {"t": 1, "k": 1, "price": UP, "value": tree["v_up"]}]}
out["q"]

## 4. Average, then discount

Weight the payoffs by $q$, average, discount: the same price as copying the option with shares and a loan (Episode 5).

In [ ]:
avg = q * tree["v_up"] + (1 - q) * tree["v_down"]
out["price"] = {"average_payoff": avg, "by_q": math.exp(-r * T) * avg, "by_copy": tree["price"]}
assert abs(out["price"]["by_q"] - out["price"]["by_copy"]) < 1e-12
out["price"]

**Any payoff.** The same q prices every payoff on this tree. The \$105 put pays nothing up and \$25 down; its q-price matches Episode 5's copy and put-call parity.

In [ ]:
put = one_step(S, UP, DOWN, K, T, r, call=False)
put_q = math.exp(-r * T) * (q * put["v_up"] + (1 - q) * put["v_down"])
out["put"] = {"pay_down": put["v_down"], "by_q": put_q, "by_copy": put["price"],
              "parity_put": tree["price"] - (S - K * math.exp(-r * T))}
assert abs(put_q - put["price"]) < 1e-12 and abs(put_q - out["put"]["parity_put"]) < 1e-12
out["put"]

## 5. Why "risk-neutral"

Under $q$ the share's average value in a year is exactly the forward, $S e^{rT}$: it grows at the risk-free rate, as if nobody asked to be paid for risk. That's the name. It's a pricing device, not a claim about anyone's preferences.

In [ ]:
out["neutral"] = {"expected_share": q * UP + (1 - q) * DOWN, "forward": S * math.exp(r * T)}
assert abs(out["neutral"]["expected_share"] - out["neutral"]["forward"]) < 1e-12
out["neutral"]

## 6. Fake odds, real price

The real-world probability $p$ is an opinion and differs between people; $q$ is forced by the two outcomes and the interest rate. The price uses only $q$.

In [ ]:
rows = [{"view": name, "p_up_pct": p * 100, "expected_share": p * UP + (1 - p) * DOWN, "price": tree["price"]}
        for name, p in (("pessimist", 0.3), ("optimist", 0.8))]
out["views"] = {"rows": rows}
pd.DataFrame(rows)

## 7. When rates change

Higher rates make the forward higher, so $q$ must rise to make the share grow faster on average. The call gets dearer.

In [ ]:
rows = []
for rate in (0.0, 0.036, 0.05):
    t_ = one_step(S, UP, DOWN, K, T, rate)
    rows.append({"rate_pct": rate * 100, "q_pct": t_["q"] * 100, "call": t_["price"]})
out["rates"] = {"rows": rows}
pd.DataFrame(rows)

## 8. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")